# Ejecutar todos los bancos de Globex

Pulsa **▶** en la única celda o **Ejecutar todo / Run All** e introduce la clave cuando se solicite.

Se enviarán **228 casos**: 99 FAQ, 24 de derivación, 35 de productos especiales, 40 cotizaciones y 30 chats. Cada chat se envía completo y separado de los demás.

Al terminar se entrega **RESPUESTAS.zip**, con un JSON por banco. En Colab se inicia la descarga; en Windows se abre el ZIP y en Jupyter aparece un enlace. No se calcula ningún KPI.

Si la ejecución se interrumpe, repetir Play conserva las capturas guardadas y continúa con los casos pendientes. Las solicitudes sin confirmación no se reenvían automáticamente.


In [ ]:
# Ejecutar una vez para enviar los 228 casos. La clave se solicita en entrada oculta.
import sys, json, hashlib, types
from pathlib import Path
if sys.version_info < (3, 10):
    raise RuntimeError("Se necesita Python 3.10+. En Windows ejecuta Preparar_y_Abrir_VSCode.cmd.")
CODIGO_EJECUTOR = '"""Ejecuta bancos reales de Globex y conserva capturas; no evalúa respuestas."""\nfrom __future__ import annotations\n\nimport getpass\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport socket\nimport ssl\nimport sys\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\nimport uuid\nimport zipfile\nfrom datetime import datetime, timezone\nfrom globex_export import export_results\n\nVERSION = "GLOBEX_EJECUTOR_FINAL_20261115"\nSCHEMA = "GLOBEX_CAPTURA_1"\nDEFAULT_URL = "https://globex-mvp.onrender.com"\nHERE = Path(__file__).resolve().parent\nBANKS = ("FAQ", "DERIVACION", "PRODUCTOS", "COTIZACIONES", "CHATS_COMPARTIDOS")\nID_RE = re.compile(r"^[A-Za-z0-9_-]{1,30}$")\n\n\ndef utc_now():\n    return datetime.now(timezone.utc).isoformat(timespec="milliseconds")\n\n\ndef canonical(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")\n\n\ndef sha(value):\n    return hashlib.sha256(value).hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    with tmp.open("w", encoding="utf-8", newline="\\n") as stream:\n        json.dump(value, stream, ensure_ascii=False, indent=2)\n        stream.write("\\n")\n        stream.flush()\n        os.fsync(stream.fileno())\n    os.replace(tmp, path)\n\n\ndef append_jsonl(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("a", encoding="utf-8", newline="\\n") as stream:\n        stream.write(json.dumps(value, ensure_ascii=False, separators=(",", ":")) + "\\n")\n        stream.flush()\n        os.fsync(stream.fileno())\n\n\ndef load_jsonl(path):\n    """Una última línea interrumpida no se confunde con una captura completa."""\n    path = Path(path)\n    if not path.exists():\n        return []\n    records = []\n    for line in path.read_text(encoding="utf-8").splitlines():\n        if not line.strip():\n            continue\n        try:\n            records.append(json.loads(line))\n        except json.JSONDecodeError as exc:\n            raise ValueError("JSONL incompleto: " + path.name + ". Conserve el archivo para recuperar la captura.") from exc\n    return records\n\n\n\n\ndef validate_bundle(bundle):\n    if sha(canonical(bundle["casos"])) != bundle["sha256_casos"]:\n        raise ValueError("El banco cambió: la huella de sus casos no coincide.")\n    seen = set()\n    for case in bundle["casos"]:\n        key = (case["banco_id"], case["id"])\n        if key in seen or not ID_RE.fullmatch(case["id"]):\n            raise ValueError("ID duplicado o inválido: " + case["id"])\n        seen.add(key)\n        if case["banco"] not in BANKS:\n            raise ValueError("Banco desconocido: " + case["banco"])\n        if ("pregunta" in case) == ("mensajes" in case):\n            raise ValueError("Cada caso necesita pregunta o mensajes, exclusivamente.")\n        messages = case.get("mensajes", [case.get("pregunta")])\n        if not isinstance(messages, list) or not 1 <= len(messages) <= 6:\n            raise ValueError("Cantidad de turnos inválida: " + case["id"])\n        if any(not isinstance(m, str) or not m.strip() or len(m) > 2000 for m in messages):\n            raise ValueError("Mensaje vacío o superior a 2000 caracteres: " + case["id"])\n    return bundle\n\n\ndef _names(value):\n    if value is None:\n        return []\n    if isinstance(value, str):\n        return [item.strip() for item in value.split(",") if item.strip()]\n    return list(value)\n\n\ndef select_cases(bundle, perfil="todos", bancos=None, universos=None, ids=None):\n    validate_bundle(bundle)\n    cases = bundle["casos"]\n    if perfil != "todos":\n        raise ValueError("Esta entrega ejecuta el banco completo.")\n    chosen_banks = set(_names(bancos))\n    chosen_ids = set(_names(ids))\n    chosen_universes = set(_names(universos))\n    if chosen_banks - set(BANKS):\n        raise ValueError("Banco desconocido: " + ", ".join(sorted(chosen_banks - set(BANKS))))\n    if any(":" not in u for u in chosen_universes):\n        raise ValueError("Cada universo debe indicar BANCO:UNIVERSO; por ejemplo FAQ:U07.")\n    if chosen_banks:\n        cases = [c for c in cases if c["banco"] in chosen_banks]\n    if chosen_universes:\n        known_universes = {c["banco"] + ":" + c["universo"] for c in bundle["casos"]}\n        if chosen_universes - known_universes:\n            raise ValueError("Universo desconocido: " + ", ".join(sorted(chosen_universes - known_universes)))\n        cases = [c for c in cases if c["banco"] + ":" + c["universo"] in chosen_universes]\n    if chosen_ids:\n        available = {c["id"] for c in cases}\n        if chosen_ids - available:\n            raise ValueError("IDs fuera de la selección: " + ", ".join(sorted(chosen_ids - available)))\n        cases = [c for c in cases if c["id"] in chosen_ids]\n    if not cases:\n        raise ValueError("La selección no contiene casos.")\n    return cases\n\n\ndef payload_for(case, corrida_id):\n    payload = {"banco_id": case["banco_id"], "corrida_id": corrida_id, "id": case["id"]}\n    if "mensajes" in case:\n        payload["mensajes"] = list(case["mensajes"])\n    else:\n        payload["pregunta"] = case["pregunta"]\n    return payload\n\n\ndef resolve_key(explicit=None):\n    if explicit:\n        key = explicit\n    else:\n        key = os.environ.get("EVAL_API_KEY") or os.environ.get("GLOBEX_EVAL_API_KEY")\n        if not key:\n            try:\n                from google.colab import userdata\n                key = userdata.get("EVAL_API_KEY")\n            except Exception:\n                key = None\n        if not key:\n            try:\n                key = getpass.getpass("Clave de evaluación Globex (entrada oculta): ")\n            except (EOFError, KeyboardInterrupt) as exc:\n                raise RuntimeError("Configure EVAL_API_KEY o el secreto de Colab antes de ejecutar.") from exc\n    key = key.strip()\n    if not key or any(c in key for c in "\\r\\n"):\n        raise ValueError("La clave está vacía o contiene saltos de línea.")\n    return key\n\n\ndef normalize_url(url):\n    url = url.rstrip("/")\n    parsed = urllib.parse.urlsplit(url)\n    if parsed.username or parsed.password or parsed.query or parsed.fragment:\n        raise ValueError("La URL del backend no debe incluir credenciales, parámetros o fragmentos.")\n    local = parsed.hostname in ("localhost", "127.0.0.1", "::1")\n    if not parsed.hostname or (parsed.scheme != "https" and not (local and parsed.scheme == "http")):\n        raise ValueError("Use HTTPS; HTTP solo se admite para un servidor local.")\n    if parsed.path not in ("", "/"):\n        raise ValueError("Indique solo la URL base, sin /api/eval/caso.")\n    return url\n\n\nclass NoRedirect(urllib.request.HTTPRedirectHandler):\n    def redirect_request(self, req, fp, code, msg, headers, newurl):\n        # Evita reenviar la clave a otra dirección y repetir un POST.\n        return None\n\n\ndef warmup_backend(base_url, timeout):\n    """GET del health check publicado; no crea sesiones ni invoca al modelo."""\n    request = urllib.request.Request(base_url + "/", headers={"Accept": "application/json", "User-Agent": VERSION})\n    try:\n        with urllib.request.build_opener(NoRedirect()).open(request, timeout=timeout) as response:\n            response.read()\n            if response.status != 200:\n                raise RuntimeError("El backend no confirmó disponibilidad.")\n    except (urllib.error.URLError, TimeoutError, OSError) as exc:\n        raise RuntimeError("No se pudo comprobar el backend; no se enviaron preguntas. " + type(exc).__name__) from exc\n    return {"http": 200, "fecha_utc": utc_now()}\n\n\ndef send_case(url, payload, key, timeout):\n    request = urllib.request.Request(\n        url, data=canonical(payload), method="POST",\n        headers={"X-API-Key": key, "Content-Type": "application/json",\n                 "Accept": "application/json", "Accept-Encoding": "identity",\n                 "User-Agent": VERSION},\n    )\n    opener = urllib.request.build_opener(NoRedirect())\n    try:\n        with opener.open(request, timeout=timeout) as response:\n            body = response.read()\n            return {"http": response.status, "body": body,\n                    "content_type": response.headers.get("Content-Type"),\n                    "retry_after": response.headers.get("Retry-After"), "error_cliente": None}\n    except urllib.error.HTTPError as exc:\n        return {"http": exc.code, "body": exc.read(),\n                "content_type": exc.headers.get("Content-Type"),\n                "retry_after": exc.headers.get("Retry-After"), "error_cliente": None}\n    except (urllib.error.URLError, TimeoutError, socket.timeout, ssl.SSLError, OSError) as exc:\n        return {"http": None, "body": b"", "content_type": None, "retry_after": None,\n                "error_cliente": {"tipo": type(exc).__name__,\n                                  "mensaje": str(exc).replace(key, "[REDACTADO]"),\n                                  "resultado_de_ejecucion_desconocido": True}}\n\n\ndef inspect_structure(response_api, payload):\n    """Observaciones de estructura, nunca calificaciones del chatbot."""\n    notes = []\n    if not isinstance(response_api, dict):\n        return ["El cuerpo no contiene un objeto JSON de la API."]\n    for field in ("id", "banco_id", "corrida_id"):\n        if response_api.get(field) != payload[field]:\n            notes.append("La identificación devuelta no coincide en " + field + ".")\n    for field in ("respuesta_bot", "turnos", "datos_recopilados", "resumen_caso",\n                  "registros_acciones", "cotizacion", "control_versiones"):\n        if field not in response_api:\n            notes.append("El endpoint no incluyó el campo " + field + ".")\n    if "turnos" in response_api and not isinstance(response_api["turnos"], list):\n        notes.append("turnos no es una lista.")\n    return notes\n\n\ndef _capture(case, payload, snapshot, capture_id, started, finished, run_dir, key):\n    raw = snapshot["body"]\n    if key.encode("utf-8") in raw:\n        raise RuntimeError("La respuesta contiene la credencial. Se detuvo la exportación para evitar divulgarla.")\n    original = None\n    original_hash = None\n    response_api = None\n    json_valid = False\n    if snapshot["http"] is not None:\n        original = "originales/" + capture_id + ".body"\n        target = run_dir / original\n        target.parent.mkdir(parents=True, exist_ok=True)\n        target.write_bytes(raw)\n        original_hash = sha(raw)\n        try:\n            response_api = json.loads(raw)\n            json_valid = True\n        except (json.JSONDecodeError, UnicodeDecodeError):\n            pass\n    if snapshot["error_cliente"]:\n        state = "SIN_CONFIRMACION_POR_ERROR_DE_CONEXION"\n    elif snapshot["http"] == 200 and json_valid:\n        state = "RESPUESTA_HTTP_JSON_RECIBIDA"\n    else:\n        state = "ERROR_HTTP_O_CUERPO_NO_JSON"\n    record = {\n        "schema_version": SCHEMA, "version_ejecutor": VERSION,\n        "id": case["id"], "banco": case["banco"], "banco_id": case["banco_id"],\n        "universo": case["universo"], "version_banco": case["version_banco"],\n        "corrida_id": payload["corrida_id"], "id_captura": capture_id, "intento": 1,\n        "fecha_inicio_utc": started, "fecha_fin_utc": finished,\n        "payload_enviado": payload, "sha256_payload": sha(canonical(payload)),\n        "http": snapshot["http"], "estado_transporte": state,\n        "content_type": snapshot["content_type"], "retry_after": snapshot["retry_after"],\n        "error_cliente": snapshot["error_cliente"], "respuesta_api": response_api,\n        "archivo_original": original, "sha256_original": original_hash,\n        "observaciones_formato": inspect_structure(response_api, payload) if json_valid else [],\n    }\n    return record\n\n\ndef _existing_records(run_dir):\n    return [item for path in sorted((run_dir / "resultados").glob("*.jsonl")) for item in load_jsonl(path)]\n\n\ndef _signature(url, bundle, cases):\n    return sha(canonical({"url": url, "banco": bundle["sha256_casos"],\n                          "casos": [(c["banco_id"], c["id"]) for c in cases]}))\n\n\ndef _plan(run_dir, bundle, cases, corrida_id, url, signature, timeout, max_per_minute):\n    manifest = run_dir / "manifiesto.json"\n    inventories = [\n        {"banco": bank, "banco_id": next(c["banco_id"] for c in cases if c["banco"] == bank),\n         "ids_previstos": [c["id"] for c in bundle["casos"] if c["banco"] == bank],\n         "ids_planificados_esta_corrida": [c["id"] for c in cases if c["banco"] == bank],\n         "seleccion_cubre_banco_completo": (\n             sum(c["banco"] == bank for c in cases) == sum(c["banco"] == bank for c in bundle["casos"]))}\n        for bank in BANKS if any(c["banco"] == bank for c in cases)\n    ]\n    if manifest.exists():\n        previous = json.loads(manifest.read_text(encoding="utf-8"))\n        if previous["firma_seleccion"] != signature or previous["corrida_id"] != corrida_id:\n            raise ValueError("La corrida existente tiene otro banco, selección o servidor. Use una nueva corrida.")\n        previous["bancos"] = inventories\n        previous["tipo_inventario_casos_previstos"] = "SOLICITUDES_PLANIFICADAS_EN_ESTA_CORRIDA"\n        return previous\n    plan = {\n        "schema_version": SCHEMA, "version_ejecutor": VERSION,\n        "corrida_id": corrida_id, "fecha_creacion_utc": utc_now(),\n        "backend": url, "endpoint": url + "/api/eval/caso",\n        "version_banco": bundle["version_paquete"], "sha256_casos": bundle["sha256_casos"],\n        "firma_seleccion": signature, "version_python": sys.version.split()[0],\n        "timeout_segundos": timeout, "limite_casos_por_minuto_configurado": max_per_minute,\n        "evaluacion_ejecutada": False,\n        "casos_previstos": [{"id": c["id"], "banco": c["banco"], "banco_id": c["banco_id"],\n                            "universo": c["universo"], "version_banco": c["version_banco"]}\n                           for c in cases],\n        "bancos": inventories,\n        "tipo_inventario_casos_previstos": "SOLICITUDES_PLANIFICADAS_EN_ESTA_CORRIDA",\n    }\n    atomic_json(manifest, plan)\n    return plan\n\n\ndef package_results(run_dir, key=None):\n    paths = sorted(p for p in run_dir.rglob("*") if p.is_file() and p.name != "MANIFIESTO_SHA256.txt"\n                   and "respuestas" not in p.relative_to(run_dir).parts)\n    checksums = []\n    for path in paths:\n        data = path.read_bytes()\n        if key and key.encode("utf-8") in data:\n            raise RuntimeError("No se exportará un archivo que contenga la credencial.")\n        checksums.append(sha(data) + "  " + path.relative_to(run_dir).as_posix())\n    (run_dir / "MANIFIESTO_SHA256.txt").write_text("\\n".join(checksums) + "\\n", encoding="utf-8")\n    zip_path = run_dir.with_name(run_dir.name + "_RESPALDO_TECNICO.zip")\n    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as archive:\n        for path in sorted(p for p in run_dir.rglob("*") if p.is_file()\n                           and "respuestas" not in p.relative_to(run_dir).parts):\n            archive.write(path, run_dir.name + "/" + path.relative_to(run_dir).as_posix())\n    with zipfile.ZipFile(zip_path) as archive:\n        if archive.testzip() is not None:\n            raise RuntimeError("El ZIP no pasó la comprobación de integridad.")\n    return zip_path\n\n\ndef run(bundle=None, perfil="todos", bancos=None, universos=None, ids=None,\n        output_base="resultados_globex", corrida_id=None, nueva_corrida=False,\n        base_url=DEFAULT_URL, timeout=180, max_per_minute=30, preparar=False,\n        api_key=None, transport=None, log=print):\n    """El transporte sustituible permite verificar errores sin llamar a Globex."""\n    if sys.version_info < (3, 10):\n        raise RuntimeError("Use Python 3.10 o superior; en Windows ejecute Preparar_y_Abrir_VSCode.cmd.")\n    if not 1 <= max_per_minute <= 60 or timeout <= 0:\n        raise ValueError("El límite debe estar entre 1 y 60 casos/minuto y el timeout debe ser positivo.")\n    if bundle is None:\n        raise ValueError("El notebook debe proporcionar su banco incorporado.")\n    cases = select_cases(bundle, perfil, bancos, universos, ids)\n    base_url = normalize_url(base_url)\n    signature = _signature(base_url, bundle, cases)\n    output_base = Path(output_base).expanduser().resolve()\n    output_base.mkdir(parents=True, exist_ok=True)\n    pointer = output_base / ("ultima_" + signature[:20] + ".json")\n    if corrida_id is None:\n        if pointer.exists() and not nueva_corrida:\n            corrida_id = json.loads(pointer.read_text(encoding="utf-8"))["corrida_id"]\n        else:\n            corrida_id = "RUN_" + datetime.now(timezone.utc).strftime("%y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:4]\n    if not ID_RE.fullmatch(corrida_id):\n        raise ValueError("corrida_id: de 1 a 30 letras, números, guiones o guiones bajos.")\n    run_dir = output_base / corrida_id\n    plan = _plan(run_dir, bundle, cases, corrida_id, base_url, signature, timeout, max_per_minute)\n    atomic_json(pointer, {"corrida_id": corrida_id, "firma_seleccion": signature})\n    log("Corrida: " + corrida_id + " | Casos seleccionados: " + str(len(cases)))\n    for bank in BANKS:\n        count = sum(c["banco"] == bank for c in cases)\n        if count:\n            log("  " + bank + ": " + str(count))\n    if preparar:\n        log("Plan guardado. No se envió ninguna solicitud.")\n        return {"corrida_id": corrida_id, "directorio": str(run_dir), "previstos": len(cases),\n                "enviados_ahora": 0, "zip": None, "solo_preparacion": True}\n    records = _existing_records(run_dir)\n    expected = {(c["banco_id"], c["id"]) for c in cases}\n    captured = {(r["banco_id"], r["id"]) for r in records}\n    if not captured <= expected or len(records) != len(captured):\n        raise ValueError("Hay capturas duplicadas o ajenas al inventario. No se elegirá la más favorable.")\n    journal = load_jsonl(run_dir / "intentos.jsonl")\n    inflight = {(r["banco_id"], r["id"]) for r in journal if r.get("evento") == "SOLICITUD_INICIADA"} - captured\n    pending_cases = [c for c in cases if (c["banco_id"], c["id"]) not in captured | inflight]\n    if inflight:\n        log("Hay solicitudes previas sin captura confirmada. No se reenviarán automáticamente.")\n    key = resolve_key(api_key) if pending_cases else None\n    if pending_cases and transport is None:\n        log("Comprobando el backend; el primer arranque de Render puede tardar...")\n        plan["comprobacion_backend"] = warmup_backend(base_url, timeout)\n        atomic_json(run_dir / "manifiesto.json", plan)\n    send = transport or send_case\n    sent = 0\n    last_start = None\n    stop_reason = None\n    try:\n        for case in pending_cases:\n            if last_start is not None:\n                remaining = 60.0 / max_per_minute - (time.monotonic() - last_start)\n                if remaining > 0:\n                    time.sleep(remaining)\n            payload = payload_for(case, corrida_id)\n            capture_id = case["id"] + "_" + uuid.uuid4().hex[:12]\n            started = utc_now()\n            append_jsonl(run_dir / "intentos.jsonl", {\n                "evento": "SOLICITUD_INICIADA", "id_captura": capture_id, "id": case["id"],\n                "banco_id": case["banco_id"], "fecha_utc": started, "sha256_payload": sha(canonical(payload)),\n            })\n            log("Enviando " + case["banco"] + " / " + case["id"] + " ...")\n            last_start = time.monotonic()\n            sent += 1\n            snapshot = send(base_url + "/api/eval/caso", payload, key, timeout)\n            record = _capture(case, payload, snapshot, capture_id, started, utc_now(), run_dir, key)\n            path = run_dir / "resultados" / (case["banco"] + ".jsonl")\n            append_jsonl(path, record)\n            records.append(record)\n            captured.add((case["banco_id"], case["id"]))\n            append_jsonl(run_dir / "intentos.jsonl", {\n                "evento": "CAPTURA_GUARDADA", "id_captura": capture_id,\n                "id": case["id"], "banco_id": case["banco_id"], "fecha_utc": utc_now(),\n            })\n            atomic_json(run_dir / "progreso.json", {\n                "corrida_id": corrida_id, "fecha_utc": utc_now(),\n                "ids_con_captura": [r["id"] for r in records],\n                "sin_calificaciones": True,\n            })\n            log("  HTTP " + str(record["http"]) + " | " + record["estado_transporte"])\n            if record["observaciones_formato"]:\n                log("  Observaciones de estructura guardadas: " + str(len(record["observaciones_formato"])))\n            if snapshot["error_cliente"] or snapshot["http"] != 200 or not isinstance(record["respuesta_api"], dict):\n                stop_reason = "INCIDENCIA_DE_TRANSPORTE_O_CONTRATO"\n                log("Se conserva la incidencia y se detiene la corrida. No hay reintentos automáticos.")\n                break\n    except KeyboardInterrupt:\n        stop_reason = "INTERRUMPIDA_POR_USUARIO"\n        log("Interrumpido. Las capturas ya recibidas se conservan.")\n    except Exception as exc:\n        stop_reason = "ERROR_LOCAL_DE_EJECUCION"\n        message = str(exc).replace(key, "[REDACTADO]") if key else str(exc)\n        atomic_json(run_dir / "error_local.json", {"tipo": type(exc).__name__, "mensaje": message, "fecha_utc": utc_now()})\n        log("Se detuvo la ejecución por un error local. Los resultados anteriores se conservan.")\n    finally:\n        journal = load_jsonl(run_dir / "intentos.jsonl")\n        inflight = {(r["banco_id"], r["id"]) for r in journal if r.get("evento") == "SOLICITUD_INICIADA"} - captured\n        commits = sorted({str(r["respuesta_api"].get("control_versiones", {}).get("commit_chatbot"))\n                          for r in records if isinstance(r.get("respuesta_api"), dict)\n                          and isinstance(r["respuesta_api"].get("control_versiones"), dict)\n                          and r["respuesta_api"]["control_versiones"].get("commit_chatbot")})\n        plan["cierre"] = {\n            "fecha_utc": utc_now(), "motivo_parada": stop_reason,\n            "ids_con_captura": [r["id"] for r in records],\n            "ids_sin_captura": [c["id"] for c in cases if (c["banco_id"], c["id"]) not in captured],\n            "ids_solicitud_sin_confirmacion": sorted(cid for _, cid in inflight),\n            "ids_con_incidencia_transporte": [r["id"] for r in records if r["estado_transporte"] != "RESPUESTA_HTTP_JSON_RECIBIDA"],\n            "commits_chatbot_observados": commits,\n            "advertencia_cambio_version": len(commits) > 1,\n            "evaluacion_ejecutada": False,\n        }\n        atomic_json(run_dir / "manifiesto.json", plan)\n        zip_path, _ = export_results(run_dir, records, plan, bundle)\n        backup_path = package_results(run_dir, key)\n    result = {"corrida_id": corrida_id, "directorio": str(run_dir), "zip": str(zip_path),\n              "previstos": len(cases), "capturas_guardadas": len(records), "enviados_ahora": sent,\n              "ids_sin_captura": plan["cierre"]["ids_sin_captura"],\n              "ids_con_incidencia_transporte": plan["cierre"]["ids_con_incidencia_transporte"],\n              "evaluacion_ejecutada": False, "zip_respaldo": str(backup_path)}\n    log("Archivo de respuestas: " + str(zip_path))\n    log("Respaldo técnico conservado: " + str(backup_path))\n    return result\n\n\ndef notebook_output_dir(use_drive=False):\n    try:\n        from google.colab import drive\n    except ImportError:\n        return Path.cwd() / "resultados_globex"\n    if use_drive:\n        drive.mount("/content/drive")\n        return Path("/content/drive/MyDrive/Globex_Pruebas/resultados")\n    return Path("/content/resultados_globex")\n\n\ndef notebook_download(result):\n    if not result.get("zip"):\n        return\n    try:\n        from google.colab import files\n        files.download(result["zip"])\n    except ImportError:\n        try:\n            from IPython.display import display, FileLink\n            target = Path(result["zip"]).resolve()\n            try:\n                label = str(target.relative_to(Path.cwd()))\n            except ValueError:\n                label = str(target)\n            display(FileLink(label, result_html_prefix="Descargar resultados: "))\n        except ImportError:\n            print("Resultados: " + result["zip"])\n\n\n\n\n'
CODIGO_EXPORTADOR = '"""Organiza las respuestas sin modificar sus textos ni asignar calificaciones."""\nfrom copy import deepcopy\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport zipfile\n\nFORMATO = "GLOBEX_JUEZ_COMPACTO_1"\nGROUPS = {"FAQ": "01_FAQ", "DERIVACION": "02_Derivacion", "PRODUCTOS": "03_Productos",\n          "COTIZACIONES": "04_Cotizaciones", "CHATS_COMPARTIDOS": "05_Chats_Recopilacion_Resumen"}\n\n\ndef _same(left, right):\n    return type(left) is type(right) and left == right\n\n\ndef _render_components(value):\n    """Solo reconoce la representación textual exacta de componentes documentados."""\n    if not isinstance(value, list):\n        return None\n    rendered = []\n    for part in value:\n        if (not isinstance(part, dict) or set(part) - {"tipo", "texto", "opciones"}\n                or part.get("tipo") not in {"texto", "cotizacion", "botones"}\n                or not isinstance(part.get("texto"), str)):\n            return None\n        text = part["texto"]\n        options = part.get("opciones", [])\n        if not isinstance(options, list) or any(not isinstance(option, str) for option in options):\n            return None\n        if options:\n            text += "\\n[Opciones: " + " | ".join(options) + "]"\n        rendered.append(text)\n    return "\\n\\n".join(rendered)\n\n\ndef _outputs(api, anchor=None):\n    result, known = {}, []\n    if anchor is not None:\n        known.append(anchor)\n    candidates = []\n    if "respuesta_bot" in api:\n        candidates.append(("respuesta_bot", api["respuesta_bot"]))\n    turns = api.get("turnos")\n    if isinstance(turns, list) and len(turns) == 1 and isinstance(turns[0], dict) and "respuesta_bot" in turns[0]:\n        candidates.append(("turnos[0].respuesta_bot", turns[0]["respuesta_bot"]))\n    for origin, value in candidates:\n        if any(_same(value, previous) for previous in known):\n            continue\n        if not known and "respuesta" not in result:\n            result["respuesta"] = deepcopy(value)\n            if origin != "respuesta_bot":\n                result["origen_respuesta"] = origin\n        else:\n            result.setdefault("otras_salidas", []).append({"origen": origin, "contenido": deepcopy(value)})\n        known.append(value)\n    if "respuestas" in api:\n        rendered = _render_components(api["respuestas"])\n        if rendered is None or not any(_same(rendered, previous) for previous in known):\n            # No se resume ni se elige una versión si los mensajes difieren.\n            result["mensajes_bot"] = deepcopy(api["respuestas"])\n    return result\n\n\ndef _summaries(api):\n    result = {"textos": []}\n    sources = [\n        ("caso", api, "resumen_caso"),\n        ("consola", api.get("registros_acciones", {}), "resumen_en_consola"),\n    ]\n    for label, container, key in sources:\n        if not isinstance(container, dict) or key not in container:\n            continue\n        value = container[key]\n        if not isinstance(value, dict):\n            result[label] = {"valor_recibido": deepcopy(value)}\n            continue\n        record = {}\n        if "publicado" in value:\n            record["publicado"] = deepcopy(value["publicado"])\n        if "texto" in value:\n            text = value["texto"]\n            if isinstance(text, str):\n                if text not in result["textos"]:\n                    result["textos"].append(text)\n                record["texto_ref"] = result["textos"].index(text)\n            else:\n                record["texto"] = deepcopy(text)\n        result[label] = record\n    return result\n\n\ndef _conversation(api, payload):\n    turns, sent = api.get("turnos"), payload.get("mensajes", [])\n    if (isinstance(turns, list) and len(turns) == len(sent)\n            and all(isinstance(t, dict) and t.get("mensaje") == m for t, m in zip(turns, sent))):\n        result = {"conversacion": [\n            {key: deepcopy(turn[key]) for key in ("mensaje", "respuesta_bot", "tipo_respuesta") if key in turn}\n            for turn in turns\n        ]}\n        last = turns[-1].get("respuesta_bot") if turns else None\n    else:\n        result = {"mensajes_enviados": deepcopy(sent)}\n        if "turnos" in api:\n            result["conversacion_recibida"] = deepcopy(turns)\n        last = None\n    result.update(_outputs(api, last))\n    if "mensajes" in api and api["mensajes"] != sent:\n        result["mensajes_devueltos_distintos"] = deepcopy(api["mensajes"])\n    return result\n\n\ndef _capture_status(record, case, plan):\n    if record is None:\n        uncertain = plan.get("cierre", {}).get("ids_solicitud_sin_confirmacion", [])\n        state = "SIN_CONFIRMACION" if case["id"] in uncertain else "SIN_CAPTURA"\n        return {"estado": state}\n    api, http = record.get("respuesta_api"), record.get("http")\n    if http == 200 and isinstance(api, dict):\n        result = {"estado": "RECIBIDA"}\n        if "ejecutado" in api:\n            result["ejecutado"] = deepcopy(api["ejecutado"])\n        if api.get("error_ejecucion") is not None:\n            result["error_ejecucion"] = deepcopy(api["error_ejecucion"])\n    else:\n        result = {"estado": "SIN_CONFIRMACION" if http is None else "ERROR_HTTP_O_FORMATO", "http": http}\n        if record.get("error_cliente"):\n            result["error_cliente"] = deepcopy(record["error_cliente"])\n    if record.get("observaciones_formato"):\n        result["incidencias_estructura"] = deepcopy(record["observaciones_formato"])\n    return result\n\n\ndef project_case(case, record, plan):\n    """Cada texto es copia literal de una ruta original, o se conserva su alternativa."""\n    result = {"id": case["id"], "universo": case["universo"],\n              "captura": _capture_status(record, case, plan)}\n    if record is None or record.get("http") != 200 or not isinstance(record.get("respuesta_api"), dict):\n        return result\n    api, payload = record["respuesta_api"], record["payload_enviado"]\n    expected = {"id": case["id"], "banco_id": case["banco_id"], "corrida_id": plan["corrida_id"]}\n    expected["mensajes" if "mensajes" in case else "pregunta"] = case.get("mensajes", case.get("pregunta"))\n    if payload != expected:\n        raise ValueError("El payload no corresponde al banco congelado: " + case["id"])\n    for name in ("id", "banco_id", "corrida_id"):\n        if api.get(name) != expected[name]:\n            result["captura"].setdefault("incidencias_identificacion", {})[name] = deepcopy(api.get(name))\n    if case["banco"] == "CHATS_COMPARTIDOS":\n        result.update(_conversation(api, payload))\n        if "datos_recopilados" in api:\n            data = deepcopy(api["datos_recopilados"])\n            if isinstance(data, dict):\n                data.pop("cotizacion_aceptada", None)  # Indicador técnico excluido de ambos KPI.\n            result["datos_recopilados"] = data\n        result["resumen"] = _summaries(api)\n        actions = api.get("registros_acciones")\n        if isinstance(actions, dict) and "bot_pausado" in actions:\n            result["bot_pausado"] = deepcopy(actions["bot_pausado"])\n    else:\n        result["pregunta"] = case["pregunta"]\n        result.update(_outputs(api))\n        if "pregunta" in api and api["pregunta"] != payload["pregunta"]:\n            result["pregunta_devuelta_distinta"] = deepcopy(api["pregunta"])\n    if case["banco"] in {"PRODUCTOS", "COTIZACIONES", "CHATS_COMPARTIDOS"} and "cotizacion" in api:\n        result["cotizacion"] = deepcopy(api["cotizacion"])\n    if case["banco"] == "PRODUCTOS":\n        actions = api.get("registros_acciones")\n        if isinstance(actions, dict) and isinstance(actions.get("herramientas"), list):\n            relevant = [\n                {key: deepcopy(tool[key]) for key in ("nombre", "estado", "resultado") if key in tool}\n                for tool in actions["herramientas"] if isinstance(tool, dict)\n                and any(name in str(tool.get("nombre", "")).lower() for name in ("cotiz", "proforma", "pdf"))\n            ]\n            if relevant:\n                result["acciones_cotizacion"] = relevant\n    return result\n\n\ndef _json_file(path, data):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(data, ensure_ascii=False, separators=(",", ":")) + "\\n", encoding="utf-8")\n\n\ndef _zip(path, files, base):\n    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as archive:\n        for file in sorted(files):\n            archive.write(file, file.relative_to(base).as_posix())\n    with zipfile.ZipFile(path) as archive:\n        if archive.testzip() is not None:\n            raise RuntimeError("ZIP de exportación no válido.")\n\n\ndef _commit(record):\n    api = record.get("respuesta_api") if record else None\n    metadata = api.get("control_versiones") if isinstance(api, dict) else None\n    return metadata.get("commit_chatbot") if isinstance(metadata, dict) else None\n\n\nIDS_BLOQUEADOS = [\'PRO002\', \'PRO006\', \'PRO009\', \'PRO012\', \'PRO015\', \'PRO018\', \'PRO020\', \'PRO022\', \'PRO024\', \'PRO026\', \'PRO032\', \'PRO035\', \'PRO036\', \'PRO041\', \'PRO045\', \'PRO049\', \'PRO051\', \'PRO053\', \'PRO058\', \'PRO061\', \'PRO066\', \'PRO067\', \'PRO068\', \'PRO069\', \'PRO071\']\n\ndef export_results(run_dir, records, plan, bundle):\n    cases_by_id = {c[\'id\']: c for c in bundle[\'casos\']}\n    record_by_id = {}\n    for record in records:\n        if record[\'id\'] in record_by_id:\n            raise ValueError(\'Capturas repetidas; no se elegirá una respuesta entre varias.\')\n        record_by_id[record[\'id\']] = record\n    dest = run_dir / \'respuestas\'\n    files = []\n    for bank in plan[\'bancos\']:\n        name = bank[\'banco\']\n        selected = bank[\'ids_planificados_esta_corrida\']\n        folder = dest / GROUPS[name]\n        result = {\n            \'formato\': FORMATO, \'banco\': name, \'banco_id\': bank[\'banco_id\'],\n            \'version_banco\': plan[\'version_banco\'], \'corrida_id\': plan[\'corrida_id\'],\n            \'ids_previstos\': bank[\'ids_previstos\'], \'ids_seleccionados\': selected,\n            \'seleccion_cubre_banco_completo\': bank[\'seleccion_cubre_banco_completo\'],\n        }\n        commits = {cid: _commit(record_by_id.get(cid)) for cid in selected}\n        if len({json.dumps(v, sort_keys=True) for v in commits.values()}) == 1:\n            result[\'commit_chatbot\'] = next(iter(commits.values()))\n        else:\n            result[\'commit_por_id\'] = commits\n        if name == \'PRODUCTOS\':\n            result[\'ids_bloqueados_previstos\'] = IDS_BLOQUEADOS\n        if name == \'CHATS_COMPARTIDOS\':\n            result[\'tipo_caso\'] = \'RESUMEN_PARA_ASESOR\'\n        result[\'casos\'] = [project_case(cases_by_id[cid], record_by_id.get(cid), plan) for cid in selected]\n        path = folder / \'Resultados.json\'\n        _json_file(path, result)\n        files.append(path)\n    archive = run_dir.with_name(run_dir.name + \'_RESPUESTAS.zip\')\n    _zip(archive, files, dest)\n    return archive, files\n'
BANCO_JSON = '{"schema_version":"GLOBEX_BANCO_EJECUCION_1","version_paquete":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","sha256_casos":"61cebdd88ebdad479c26c126e54ccad1840c8e46a4344a943e4669f8ae47c540","casos":[{"id":"FAQ001","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U01","pregunta":"¿Cuál es la dirección completa del almacén de Globex en Miami para enviar mi compra?"},{"id":"FAQ002","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U01","pregunta":"Quiero mandar mi compra a Globex, ¿me pasas la dirección completa de Miami?"},{"id":"FAQ003","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U01","pregunta":"La tienda pide Address line 1 y Address line 2, ¿qué pongo en cada campo para enviar a Globex?"},{"id":"FAQ004","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U01","pregunta":"¿Cómo distribuyo la calle y la suite de Globex entre dirección principal y apartamento o suite?"},{"id":"FAQ005","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U01","pregunta":"¿Qué ciudad, estado, país y código postal pongo para el almacén de Miami?"},{"id":"FAQ006","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U01","pregunta":"La tienda me pide City, State, Country y ZIP, ¿cuáles son los datos de Globex?"},{"id":"FAQ007","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U01","pregunta":"¿Qué teléfono debo colocar para que la tienda entregue mi compra en el almacén de Miami?"},{"id":"FAQ008","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U01","pregunta":"Escribí 3312 como ZIP del almacén de Miami, ¿está bien o cuál debo usar?"},{"id":"FAQ009","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U01","pregunta":"Puse 7270 NW 35TH TERRACE pero olvidé la suite, ¿qué dato debo agregar?"},{"id":"FAQ010","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U01","pregunta":"¿Uso la dirección del almacén de Globex como dirección de envío o como mi dirección personal de facturación?"},{"id":"FAQ011","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U02","pregunta":"¿Qué significa que Globex me pida poner GX junto a mi nombre?"},{"id":"FAQ012","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U02","pregunta":"¿Puedo escribir GX Nombre Apellido o Nombre Apellido GX?"},{"id":"FAQ013","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U02","pregunta":"¿GX tiene que ir antes del nombre o también puede ir después?"},{"id":"FAQ014","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U02","pregunta":"El formulario separa nombre y apellido, ¿tengo que poner GX en los dos campos?"},{"id":"FAQ015","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U02","pregunta":"¿Es obligatorio repetir GX junto al nombre y también junto al apellido?"},{"id":"FAQ016","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U02","pregunta":"¿Puedo escribir solo GX y dejar vacío mi nombre y apellido?"},{"id":"FAQ017","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U02","pregunta":"¿Para el destinatario basta con poner Globex o debo mantener mi nombre y apellido?"},{"id":"FAQ018","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U02","pregunta":"¿Escribir GX en el destinatario significa que Globex ya recibió y registró mi paquete?"},{"id":"FAQ019","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"¿Cómo funciona Globex para traer mi compra desde Estados Unidos hasta Perú?"},{"id":"FAQ020","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"Quiero comprar en eBay USA, ¿cuáles son los pasos generales para traerlo con Globex?"},{"id":"FAQ021","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"En el servicio courier normal, ¿yo compro por internet o Globex hace la compra?"},{"id":"FAQ022","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"¿Quién compra y paga el producto en la tienda cuando uso el courier estándar?"},{"id":"FAQ023","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"¿Globex vende el producto que veo en Amazon o solo presta el servicio para traerlo?"},{"id":"FAQ024","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"¿Qué hago para que una compra de una tienda de Estados Unidos llegue a Globex?"},{"id":"FAQ025","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"¿La entrega desde la tienda al almacén de Miami la coordina la tienda o Globex recoge en cualquier tienda?"},{"id":"FAQ026","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"Si la tienda de Estados Unidos no tiene envío directo a Perú, ¿igual puedo usar Globex?"},{"id":"FAQ027","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"¿Los productos de una misma cotización pueden reunirse para traerlos juntos?"},{"id":"FAQ028","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"Si los productos de mi cotización llegan por separado a Miami, ¿cómo los reúnen?"},{"id":"FAQ029","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"Todavía no he comprado nada, ¿puedo pedir información para usar Globex?"},{"id":"FAQ030","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"¿Que una marca sea estadounidense basta para saber que la compra sale de Estados Unidos?"},{"id":"FAQ031","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"¿Puedo comprar en cualquier página de Estados Unidos y traerlo con Globex?"},{"id":"FAQ032","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"¿Tiene que ser Amazon o eBay, o puedo comprar en otra página de USA?"},{"id":"FAQ033","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"La página donde quiero comprar es de Estados Unidos, ¿puedo enviarlo a su almacén de Miami?"},{"id":"FAQ034","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"¿Hacen envíos directamente de China a Perú?"},{"id":"FAQ035","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"Quiero traer un paquete de España a Perú, ¿ustedes hacen ese envío?"},{"id":"FAQ036","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"¿Puedo enviar un paquete de Perú hacia Estados Unidos con este servicio?"},{"id":"FAQ037","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U05","pregunta":"¿Dónde reviso si la tienda me está cobrando TAX antes de pagar?"},{"id":"FAQ038","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U05","pregunta":"Ya pagué mi compra, ¿dónde puedo ver cuánto TAX cobró la tienda?"},{"id":"FAQ039","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U05","pregunta":"La tienda me da un precio final con TAX incluido, ¿debo sumarlo otra vez?"},{"id":"FAQ040","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U05","pregunta":"Mi checkout ya incluye el impuesto, ¿tengo que añadir ese mismo impuesto nuevamente al precio?"},{"id":"FAQ041","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U05","pregunta":"¿Por qué el precio anunciado por una tienda puede diferir del total que aparece en el checkout?"},{"id":"FAQ042","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U05","pregunta":"Cambié la dirección de entrega antes de pagar, ¿debo revisar otra vez el TAX del checkout?"},{"id":"FAQ043","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U06","pregunta":"¿Qué es el tracking de una compra por internet?"},{"id":"FAQ044","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U06","pregunta":"¿Quién proporciona el número de seguimiento de mi compra y para qué sirve?"},{"id":"FAQ045","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U06","pregunta":"¿El número de orden de la tienda siempre es el mismo que el tracking?"},{"id":"FAQ046","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U06","pregunta":"¿Dónde encuentro el tracking que debo informar a Globex?"},{"id":"FAQ047","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U06","pregunta":"La tienda dividió mi compra en dos paquetes con dos trackings, ¿cuál debo informar?"},{"id":"FAQ048","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U06","pregunta":"Tengo tres números de seguimiento para productos de la misma cotización, ¿envío solo uno?"},{"id":"FAQ049","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U06","pregunta":"¿Qué documentos debo enviar para embarcar mi compra y por qué canal?"},{"id":"FAQ050","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U06","pregunta":"¿La documentación del embarque se envía por correo o basta con mandarla por WhatsApp?"},{"id":"FAQ051","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U06","pregunta":"Después de enviar los documentos al correo oficial de embarques, ¿debo avisar por el chat?"},{"id":"FAQ052","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U06","pregunta":"¿Enviar el correo con los documentos significa automáticamente que ya fueron aprobados?"},{"id":"FAQ053","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"Vivo en provincia, ¿puedo recibir una compra que llegue con Globex?"},{"id":"FAQ054","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"Estoy en Ilo, ¿cómo puedo recibir una compra que traiga con Globex?"},{"id":"FAQ055","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"¿Cómo pasa mi paquete del almacén de Miami hasta la agencia Shalom de mi provincia?"},{"id":"FAQ056","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"¿Qué datos debo proporcionar para preparar mi despacho por Shalom?"},{"id":"FAQ057","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"Para un envío por Shalom, ¿basta con indicar que estoy en Arequipa o debo dar la agencia exacta?"},{"id":"FAQ058","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"¿Por qué debo indicar el nombre exacto de la agencia Shalom y no solamente mi ciudad?"},{"id":"FAQ059","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"Si mi envío a provincia va por Shalom, ¿lo recojo en una agencia o lo llevan directamente a mi casa?"},{"id":"FAQ060","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"¿Qué incluye la preparación y traslado de Globex hasta la agencia Shalom?"},{"id":"FAQ061","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"¿El cargo de Globex por llevar mi paquete hasta Shalom ya incluye el transporte hasta mi provincia?"},{"id":"FAQ062","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"¿Cuándo y dónde se paga el transporte de Shalom?"},{"id":"FAQ063","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"El flyer muestra una agencia de Yurimaguas, ¿eso significa que todos tenemos que recoger allí?"},{"id":"FAQ064","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"¿Pueden enviar mi paquete a provincia por Olva?"},{"id":"FAQ065","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"Prefiero Marvisur, ¿trabajan con ese operador para provincia?"},{"id":"FAQ066","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"¿Puedo elegir otra empresa de transporte para provincia en lugar de Shalom?"},{"id":"FAQ067","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"Mi paquete pesa 4 kg, ¿cuánto cuesta la preparación y traslado de Globex hasta la agencia Shalom, sin contar el transporte de Shalom?"},{"id":"FAQ068","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"Mi paquete pesa 8 kg, ¿cuánto cuesta la preparación y traslado de Globex hasta la agencia Shalom, sin contar el transporte de Shalom?"},{"id":"FAQ069","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"Mi paquete pesa 15 kg, ¿cuánto cuesta la preparación y traslado de Globex hasta la agencia Shalom, sin contar el transporte de Shalom?"},{"id":"FAQ070","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"Mi paquete pesa 25 kg, ¿cuánto cuesta la preparación y traslado de Globex hasta la agencia Shalom, sin contar el transporte de Shalom?"},{"id":"FAQ071","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U08","pregunta":"¿Cuál es el plazo general del courier de Globex desde que llega la compra a Miami?"},{"id":"FAQ072","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U08","pregunta":"Cuando Globex menciona cinco días para el courier, ¿son hábiles o calendario?"},{"id":"FAQ073","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U08","pregunta":"¿El plazo del courier empieza cuando compro en la tienda o cuando el paquete llega al almacén de Miami?"},{"id":"FAQ074","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U08","pregunta":"¿El tiempo de la tienda hasta Miami es el mismo que el courier desde Miami hasta Perú?"},{"id":"FAQ075","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U08","pregunta":"¿Los cinco días hábiles son una garantía de que cualquier compra llegará exactamente en ese plazo?"},{"id":"FAQ076","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U08","pregunta":"¿Cuál es la dirección para recoger mi paquete en Lima?"},{"id":"FAQ077","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U08","pregunta":"En general, ¿que la tienda marque un paquete como entregado significa automáticamente que Globex ya lo registró?"},{"id":"FAQ078","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U08","pregunta":"Para enviar mi compra al almacén de Miami, ¿debo organizar yo un horario especial o la tienda coordina la entrega?"},{"id":"FAQ079","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U08","pregunta":"¿Globex tiene delivery a domicilio en Lima?"},{"id":"FAQ080","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U08","pregunta":"¿El delivery en Lima tiene una tarifa igual para todas las zonas?"},{"id":"FAQ081","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U09","pregunta":"¿El producto que compro en Amazon se lo pago a Globex o a Amazon?"},{"id":"FAQ082","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U09","pregunta":"Si yo hago la compra por internet, ¿a quién pago el precio del producto?"},{"id":"FAQ084","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U09","pregunta":"¿El pago a la tienda, el servicio de Globex y el transporte de Shalom son el mismo pago?"},{"id":"FAQ085","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U09","pregunta":"Si pago el producto por internet, ¿con eso ya están cubiertos Globex y Shalom?"},{"id":"FAQ086","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U09","pregunta":"¿Qué medios de pago acepta Globex para su servicio?"},{"id":"FAQ087","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U09","pregunta":"¿La factura de mi compra en la tienda es el mismo comprobante del pago del servicio de Globex?"},{"id":"FAQ088","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U09","pregunta":"En general, ¿una captura de transferencia prueba por sí sola que Globex ya acreditó el dinero?"},{"id":"FAQ089","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U09","pregunta":"¿Que Amazon marque mi compra como pagada significa que ya pagué el servicio de Globex?"},{"id":"FAQ090","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U09","pregunta":"Tengo el comprobante del producto de eBay, ¿eso demuestra también el pago de la logística de Globex?"},{"id":"FAQ091","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U11","pregunta":"¿Cuánto cuesta el flete courier de Globex por kilogramo?"},{"id":"FAQ092","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U11","pregunta":"¿La tarifa estándar del flete de Estados Unidos a Perú es de cuántos dólares por kilo?"},{"id":"FAQ093","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U11","pregunta":"¿Cuánto cuesta el desaduanaje y se cobra por producto?"},{"id":"FAQ094","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U11","pregunta":"Mi cotización tiene varios productos, ¿cuánto cuesta el desaduanaje y se cobra una vez o por cada producto?"},{"id":"FAQ095","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U11","pregunta":"Mi paquete pesa 0,4 kg, ¿se cobra ese peso o existe un mínimo?"},{"id":"FAQ096","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U11","pregunta":"Si la compra pesa menos de un kilo, ¿cuál es el peso mínimo que cobra Globex?"},{"id":"FAQ097","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U11","pregunta":"Si mi paquete pesa 1,2 kg, ¿lo redondean a 2 kg?"},{"id":"FAQ098","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U11","pregunta":"Para un peso de 1,05 kg, ¿cobran el peso fraccionado o el siguiente kilo completo?"},{"id":"FAQ099","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U11","pregunta":"¿El peso que estimo antes de comprar es necesariamente el peso que cobrarán al final?"},{"id":"FAQ100","banco":"FAQ","banco_id":"FAQ_99_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U11","pregunta":"¿El cobro final usa mi peso estimado o el peso confirmado del paquete?"},{"id":"DER001","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"Quiero hablar con un asesor de Globex, por favor."},{"id":"DER002","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"Necesito que me atienda una persona, pásame con un asesor."},{"id":"DER003","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"Quisiera solicitar una llamada de un asesor para mi consulta."},{"id":"DER004","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"Quiero contratar el servicio de viajero para traer una compra desde Estados Unidos."},{"id":"DER005","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"¿Me puedes cotizar la modalidad viajero de Estados Unidos a Perú?"},{"id":"DER006","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"Necesito que revisen mi compra para traerla por viajero."},{"id":"DER007","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"Quiero solicitar reempaque para mis productos."},{"id":"DER008","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"¿Qué modalidades y cargos tienen para reempacar mis paquetes?"},{"id":"DER009","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"Necesito que revisen cómo reempacar los productos de mi cotización."},{"id":"DER010","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"Quiero cotizar el courier Estados Unidos–Perú de una compra de USD 2.100, con TAX cero confirmado."},{"id":"DER011","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"Para una cotización de Estados Unidos a Perú, el producto cuesta USD 1.900 y el TAX confirmado es USD 133."},{"id":"DER012","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"Quiero una misma cotización desde Estados Unidos a Perú con dos productos: uno de USD 1.200 y otro de USD 900, ambos con TAX cero confirmado."},{"id":"DER013","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"¿Pueden revisar si mi paquete ya llegó al almacén de Miami?"},{"id":"DER014","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"Necesito que confirmen si mi pedido ya está en Lima."},{"id":"DER015","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"Quiero saber si ya embarcaron mi paquete, necesito una revisión del estado real."},{"id":"DER016","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"Mi paquete llegó dañado y necesito que revisen mi reclamo."},{"id":"DER017","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"Falta un producto dentro de mi paquete, quiero que revisen lo ocurrido."},{"id":"DER018","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"Reporto un paquete perdido y necesito atención para mi caso."},{"id":"DER019","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"Quiero negociar un descuento sobre la tarifa del courier."},{"id":"DER020","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"Me ofrecieron una tarifa de USD 7 por kilo, necesito que la validen para mi cotización."},{"id":"DER021","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"¿Pueden revisar una tarifa especial por el volumen que voy a contratar?"},{"id":"DER022","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"Quiero que Globex compre el producto por mí en la tienda de Estados Unidos."},{"id":"DER023","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"La tienda solo acepta tarjeta estadounidense, necesito que ustedes realicen la compra por mí."},{"id":"DER024","banco":"DERIVACION","banco_id":"DERIVACION_24_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U10","pregunta":"No quiero hacer yo el pago en Amazon, quiero consultar si Globex puede comprar por encargo."},{"id":"PRO002","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Quiero cotizar una pistola real de USD 300 y 1 kg, ¿me dan el precio para traerla?"},{"id":"PRO006","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Tengo una caja de municiones de USD 80 y 0,5 kg, cotízame el envío a Perú."},{"id":"PRO009","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Quiero cotizar dos chupones de bebé nuevos de Amazon, cuestan USD 12 y pesan 0,2 kg."},{"id":"PRO012","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Cotízame tres biberones de USD 30 en total y 0,6 kg, el precio ya incluye TAX."},{"id":"PRO015","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Tengo un pegamento para pestañas de USD 10 y 0,1 kg, ¿me cotizas el courier?"},{"id":"PRO018","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Quiero cotizar un Android TV Box de USD 45 y 0,4 kg desde Estados Unidos."},{"id":"PRO020","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"No necesito que lo aprueben todavía, solo cotícenme el envío de un TV Box."},{"id":"PRO022","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Quiero cotizar un dispositivo médico nuevo de USD 150 y 1 kg."},{"id":"PRO024","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Tengo receta para un artículo médico, ¿eso permite que me coticen su importación?"},{"id":"PRO026","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Quiero traer dos casacas usadas, ¿las pueden importar?"},{"id":"PRO027","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Quiero traer dos camisetas nuevas para uso personal, ¿cumplen la condición del estado de la ropa?"},{"id":"PRO031","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Mi repuesto de auto es nuevo, ¿cumple la condición de estado que exigen?"},{"id":"PRO032","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Cotiza un repuesto de auto usado de USD 60 y 1,2 kg, ya tengo los datos completos."},{"id":"PRO034","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Son cuatro juguetes nuevos para el mismo consignatario, ¿cumplen esas dos condiciones?"},{"id":"PRO035","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Quiero importar cinco juguetes nuevos para el mismo consignatario, ¿pueden hacerlo?"},{"id":"PRO036","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Tengo dos juguetes usados, ¿los pueden traer?"},{"id":"PRO040","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Quiero traer cuatro unidades de maquillaje para un mismo DNI, ¿estoy dentro del cupo?"},{"id":"PRO041","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Son cinco unidades de maquillaje para el mismo DNI, cotízame la importación."},{"id":"PRO044","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Son tres cremas de la misma marca, ¿la cantidad cumple el máximo?"},{"id":"PRO045","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Quiero traer cuatro cremas de una marca distinta a The Ordinary, ¿me las cotizan?"},{"id":"PRO048","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Tengo cuatro perfumes para el mismo DNI, ¿la cantidad entra en el límite?"},{"id":"PRO049","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Quiero cotizar cinco perfumes para el mismo DNI, todos son para uso personal."},{"id":"PRO051","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"¿Puedo traer cuatro cremas The Ordinary porque son de esa marca?"},{"id":"PRO053","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Tengo dos cremas de una marca y dos de The Ordinary, ¿puedo contarlas por separado?"},{"id":"PRO055","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Son seis figuras de colección, ¿la regla de cantidad permite superar cuatro?"},{"id":"PRO057","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"¿Cuántos shampoos puedo incluir como máximo en una cotización?"},{"id":"PRO058","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Quiero cotizar tres shampoos en la misma cotización, ¿se puede?"},{"id":"PRO060","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Son cinco pares de calzado para el mismo consignatario, ¿cumplen la regla de cantidad?"},{"id":"PRO061","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Quiero traer seis pares de calzado para el mismo consignatario, ¿me los pueden cotizar?"},{"id":"PRO064","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"No quiero traer un TV Box, lo descarté. Ahora solo quiero dos camisetas nuevas, ¿cumplen la condición de la ropa?"},{"id":"PRO066","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"¿Qué necesito saber sobre la receta para traer medicinas?"},{"id":"PRO067","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"¿Para traer vitaminas se requiere receta y quién la valida?"},{"id":"PRO068","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Quiero información para importar proteína en polvo, ¿se exige receta?"},{"id":"PRO069","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"¿Qué condición documental manejan para importar suplementos?"},{"id":"PRO071","banco":"PRODUCTOS","banco_id":"PRODUCTOS_35_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"MER","pregunta":"Ya tengo receta para un suplemento de USD 50 y 0,5 kg, cotízalo sin esperar que revisen la receta."},{"id":"U01-01","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U01","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para un libro impreso nuevo de Amazon, una unidad para uso personal. El precio del producto es USD 40,00 y el checkout confirma TAX de USD 0,00. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U01-02","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U01","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para una billetera de tela nueva de eBay, una unidad para uso personal. El precio del producto es USD 60,00 y el checkout confirma TAX de USD 0,00. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U01-03","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U01","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para una mochila textil nueva de Amazon, una unidad para uso personal. El precio del producto es USD 90,00 y el checkout confirma TAX de USD 0,00. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U01-04","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U01","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para una casaca nueva de eBay, una unidad para uso personal. El precio del producto es USD 120,00 y el checkout confirma TAX de USD 0,00. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U01-05","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U01","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para un par de zapatillas nuevas de Amazon, una unidad para uso personal. El precio del producto es USD 150,00 y el checkout confirma TAX de USD 0,00. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U02-01","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U02","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para un libro impreso nuevo de Amazon, una unidad para uso personal. El precio final de la compra es USD 53,00 y ya incluye USD 3,00 de TAX. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U02-02","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U02","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para una billetera de tela nueva de eBay, una unidad para uso personal. El precio final de la compra es USD 85,00 y ya incluye USD 5,00 de TAX. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U02-03","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U02","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para una mochila textil nueva de Amazon, una unidad para uso personal. El precio final de la compra es USD 106,00 y ya incluye USD 6,00 de TAX. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U02-04","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U02","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para una casaca nueva de eBay, una unidad para uso personal. El precio final de la compra es USD 128,40 y ya incluye USD 8,40 de TAX. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U02-05","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U02","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para un par de zapatillas nuevas de Amazon, una unidad para uso personal. El precio final de la compra es USD 159,00 y ya incluye USD 9,00 de TAX. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U03-01","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para un libro impreso nuevo de Amazon, una unidad para uso personal. El producto cuesta USD 50,00 sin TAX y el checkout confirma USD 2,50 adicionales de TAX. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U03-02","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para una billetera de tela nueva de eBay, una unidad para uso personal. El producto cuesta USD 80,00 sin TAX y el checkout confirma USD 4,80 adicionales de TAX. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U03-03","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para una mochila textil nueva de Amazon, una unidad para uso personal. El producto cuesta USD 100,00 sin TAX y el checkout confirma USD 8,00 adicionales de TAX. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U03-04","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para una casaca nueva de eBay, una unidad para uso personal. El producto cuesta USD 140,00 sin TAX y el checkout confirma USD 7,00 adicionales de TAX. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U03-05","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U03","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para un par de zapatillas nuevas de Amazon, una unidad para uso personal. El producto cuesta USD 180,00 sin TAX y el checkout confirma USD 9,00 adicionales de TAX. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U04-01","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U04","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para un libro impreso nuevo de Amazon, una unidad para uso personal. El producto cuesta USD 50,00 sin TAX, no cuento con el monto de los taxes ni con el precio final con TAX. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U04-02","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U04","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para una billetera de tela nueva de eBay, una unidad para uso personal. El producto cuesta USD 80,00 sin TAX, no cuento con el monto de los taxes ni con el precio final con TAX. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U04-03","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U04","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para una mochila textil nueva de Amazon, una unidad para uso personal. El producto cuesta USD 100,00 sin TAX, no cuento con el monto de los taxes ni con el precio final con TAX. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U04-04","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U04","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para una casaca nueva de eBay, una unidad para uso personal. El producto cuesta USD 150,00 sin TAX, no cuento con el monto de los taxes ni con el precio final con TAX. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U04-05","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U04","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para un par de zapatillas nuevas de Amazon, una unidad para uso personal. El producto cuesta USD 180,00 sin TAX, no cuento con el monto de los taxes ni con el precio final con TAX. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U05-01","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U05","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para un libro impreso nuevo de Amazon, una unidad para uso personal. El precio final confirmado es USD 40,00 con TAX incluido, pero no tengo el importe del impuesto por separado. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U05-02","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U05","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para una billetera de tela nueva de eBay, una unidad para uso personal. El precio final confirmado es USD 75,00 con TAX incluido, pero no tengo el importe del impuesto por separado. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U05-03","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U05","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para una mochila textil nueva de Amazon, una unidad para uso personal. El precio final confirmado es USD 110,00 con TAX incluido, pero no tengo el importe del impuesto por separado. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U05-04","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U05","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para una casaca nueva de eBay, una unidad para uso personal. El precio final confirmado es USD 140,00 con TAX incluido, pero no tengo el importe del impuesto por separado. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U05-05","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U05","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para un par de zapatillas nuevas de Amazon, una unidad para uso personal. El precio final confirmado es USD 190,00 con TAX incluido, pero no tengo el importe del impuesto por separado. El peso con embalaje es 1,00 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U06-01","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U06","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para un libro impreso nuevo de Amazon, una unidad para uso personal. El producto cuesta USD 30 y el checkout confirma TAX de USD 0. El peso con embalaje es 0,10 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U06-02","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U06","pregunta":"Quiero cotizar el courier de Estados Unidos a Perú para una billetera de tela nueva de eBay, una unidad para uso personal. El precio final de la compra es USD 53 y ya incluye USD 3 de TAX. El peso con embalaje es 0,40 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U06-03","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U06","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para una mochila textil nueva de Amazon, una unidad para uso personal. El producto cuesta USD 75 sin TAX y el checkout confirma USD 4,50 adicionales de TAX. El peso con embalaje es 0,70 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U06-04","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U06","pregunta":"Necesito cotizar el courier de Estados Unidos a Perú para una casaca nueva de eBay, una unidad para uso personal. El producto cuesta USD 100 sin TAX, pero no cuento con el monto de los taxes ni con el precio final con TAX. El peso con embalaje es 0,90 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U06-05","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U06","pregunta":"Quiero cotizar el courier de Estados Unidos a Perú para un par de zapatillas nuevas de Amazon para uso personal. El precio final confirmado es USD 140 con TAX incluido, pero no tengo el importe del impuesto por separado. El peso con embalaje es 0,99 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U07-01","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para un libro impreso nuevo de Amazon, una unidad para uso personal. El producto cuesta USD 50 y el checkout confirma TAX de USD 0. El peso con embalaje es 1,10 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U07-02","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"Necesito cotizar el courier de Estados Unidos a Perú para una billetera de tela nueva de eBay, una unidad para uso personal. El producto cuesta USD 80 sin TAX y el checkout confirma USD 4,80 adicionales de TAX. El peso con embalaje es 1,25 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U07-03","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"Quiero cotizar el courier de Estados Unidos a Perú para una mochila textil nueva de Amazon, una unidad para uso personal. El precio final de la compra es USD 108 y ya incluye USD 8 de TAX. El peso con embalaje es 1,50 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U07-04","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para una casaca nueva de eBay, una unidad para uso personal. El producto cuesta USD 120 sin TAX, pero no cuento con el monto de los taxes ni con el precio final con TAX. El peso con embalaje es 2,20 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U07-05","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U07","pregunta":"Necesito cotizar el courier de Estados Unidos a Perú para un par de zapatillas nuevas de Amazon para uso personal. El precio final confirmado es USD 150 con TAX incluido, pero no tengo el importe del impuesto por separado. El peso con embalaje es 2,50 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U08-01","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U08","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para un libro impreso nuevo de Amazon, una unidad para uso personal. El producto cuesta USD 195 sin TAX y el checkout confirma USD 15 adicionales de TAX. El peso con embalaje es 0,60 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U08-02","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U08","pregunta":"Quiero cotizar el courier de Estados Unidos a Perú para una billetera de tela nueva de eBay, una unidad para uso personal. El producto cuesta USD 190 sin TAX, pero no cuento con el monto de los taxes ni con el precio final con TAX. El peso con embalaje es 1,25 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U08-03","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U08","pregunta":"Necesito cotizar el courier de Estados Unidos a Perú para una mochila textil nueva de Amazon, una unidad para uso personal. El producto cuesta USD 250 sin TAX, pero no tengo el monto de los taxes ni el precio final con TAX. El peso con embalaje es 0,80 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U08-04","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U08","pregunta":"Hola, cotízame el courier de Estados Unidos a Perú para una casaca nueva de eBay, una unidad para uso personal. El producto cuesta USD 300 sin TAX y todavía no cuento con el monto de los taxes ni con el precio final con TAX. El peso con embalaje es 2,20 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"U08-05","banco":"COTIZACIONES","banco_id":"COTIZACIONES_40_CONSOLIDADO_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"U08","pregunta":"Quiero cotizar el courier de Estados Unidos a Perú para un par de zapatillas nuevas de Amazon para uso personal. El precio final confirmado de la compra es USD 210 y ya incluye USD 20 de TAX. El peso con embalaje es 1,75 kg. El envío al almacén de Globex en Miami es gratuito y recogeré en su local de Lima."},{"id":"CHAT001","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH01","mensajes":["Hola, soy Ana Torres. Quiero cotizar una billetera nueva de Amazon, una unidad. El precio final es USD 40 con TAX confirmado de USD 0 incluido.","La billetera pesa 0,50 kg con embalaje.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT002","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH01","mensajes":["Soy Bruno Salas. Quiero cotizar un libro impreso nuevo de eBay, una unidad por USD 35 finales con TAX confirmado de USD 0 incluido.","El libro pesa 0,70 kg con embalaje.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT003","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH01","mensajes":["Soy Carla Rojas. Quiero cotizar una mochila textil nueva de Amazon, una unidad.","El precio final es USD 60 con TAX confirmado de USD 0 incluido.","Pesa 0,80 kg con embalaje.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT004","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH01","mensajes":["Soy Daniel Vega. Quiero cotizar una correa nueva de eBay, una unidad. Pesa 0,30 kg con embalaje.","El precio final es USD 30 con TAX confirmado de USD 0 incluido.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT005","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH01","mensajes":["Quiero cotizar una casaca nueva de Amazon, una unidad por USD 75 finales con TAX confirmado de USD 0 incluido.","Mi nombre es Elena Ríos.","La casaca pesa 1,20 kg con embalaje.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT006","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH02","mensajes":["Soy Fabio Luna. Quiero cotizar una billetera nueva de Amazon, una unidad por USD 45 finales con TAX de USD 0 incluido y peso 0,60 kg con embalaje.","Corrijo solo el precio: es USD 65 finales. El TAX sigue siendo USD 0 y lo demás no cambia.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT007","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH02","mensajes":["Soy Gabriela Díaz. Quiero cotizar un libro impreso nuevo de eBay, una unidad por USD 28 finales, TAX de USD 0 incluido y peso 0,40 kg con embalaje.","Corrijo el peso: es 0,90 kg con embalaje, no 0,40 kg. Lo demás no cambia.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT008","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH02","mensajes":["Soy Hugo Vera. Quiero cotizar una camiseta nueva de Amazon. El precio es USD 20 por unidad con TAX de USD 0 incluido y cada camiseta pesa 0,20 kg con embalaje.","En lugar de una quiero dos camisetas iguales. Siguen siendo USD 20 y 0,20 kg por unidad.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT009","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH02","mensajes":["Soy Inés León. Quiero cotizar una correa nueva de Amazon, una unidad por USD 40 finales, TAX de USD 0 incluido y peso 0,30 kg con embalaje.","Corrijo la tienda: la compraré en eBay. El producto, precio y peso no cambian.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT010","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH02","mensajes":["Mi nombre es Jorge Paz. Quiero cotizar un libro impreso nuevo de Amazon, una unidad por USD 32 finales con TAX de USD 0 incluido y peso 0,50 kg con embalaje.","Corrijo mi apellido: mi nombre completo es Jorge Peña.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT011","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH03","mensajes":["Soy Karina Soto. Quiero cotizar una billetera nueva de eBay, una unidad de USD 55 antes de TAX. El checkout confirma TAX adicional de USD 0. Pesa 0,60 kg con embalaje.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT012","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH03","mensajes":["Soy Luis Acosta. Quiero cotizar un libro impreso nuevo de Amazon, una unidad de USD 50 sin TAX más USD 2,50 de TAX confirmado. Pesa 0,80 kg con embalaje.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT013","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH03","mensajes":["Soy Marina Vidal. Quiero cotizar una mochila textil nueva de Amazon, una unidad. El precio FINAL es USD 64 y ya incluye USD 4 de TAX. Pesa 0,90 kg con embalaje.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT014","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH03","mensajes":["Soy Nicolás Gil. Quiero cotizar una correa nueva de eBay, una unidad. El precio final confirmado es USD 45 con TAX incluido, pero desconozco el importe del TAX por separado. Pesa 0,40 kg con embalaje.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT015","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH03","mensajes":["Soy Olivia Mora. Quiero cotizar una billetera nueva de Amazon, una unidad de USD 70 antes de TAX. Todavía no conozco el TAX real ni el precio final del checkout. Pesa 0,70 kg con embalaje.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT016","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH04","mensajes":["Soy Pablo Sanz. Quiero cotizar una billetera nueva de Amazon, una unidad por USD 40 finales con TAX de USD 0 incluido. No conozco el peso; prefiero no estimarlo y dejarlo pendiente.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT017","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH04","mensajes":["Soy Raquel Costa. Quiero traer un libro impreso nuevo de eBay, una unidad de 0,50 kg con embalaje. Aún no conozco el precio ni el TAX y deben quedar pendientes.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT018","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH04","mensajes":["Soy Sergio Pino. Quiero cotizar una mochila textil nueva, una unidad por USD 60 finales con TAX confirmado de USD 0 incluido y peso 0,80 kg con embalaje. Aún no he elegido la tienda de Estados Unidos.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT019","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH04","mensajes":["Quiero cotizar una billetera nueva de Amazon, una unidad por USD 45 finales con TAX confirmado de USD 0 incluido y peso 0,60 kg con embalaje.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT020","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH04","mensajes":["Soy Teresa Campos. Quiero cotizar billeteras nuevas de eBay. Cada una cuesta USD 30 finales con TAX de USD 0 incluido y pesa 0,25 kg con embalaje. Todavía no sé cuántas compraré; deja la cantidad pendiente.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT021","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH05","mensajes":["Soy Úrsula Peña. Quiero una cotización con un libro impreso nuevo de Amazon de USD 20 y 0,40 kg, y una billetera nueva de eBay de USD 35 y 0,30 kg. Es una unidad de cada producto y ambos precios son finales con TAX de USD 0 incluido.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT022","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH05","mensajes":["Soy Valeria Núñez. Quiero cotizar una mochila textil nueva de Amazon, una unidad de USD 40 finales con TAX de USD 0 incluido y peso 0,60 kg con embalaje.","Añade a la misma cotización un cuaderno nuevo de eBay, una unidad de USD 15 finales con TAX de USD 0 incluido y peso 0,30 kg con embalaje.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT023","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH05","mensajes":["Soy Walter Ruiz. Quiero cotizar un libro impreso nuevo de Amazon de USD 25 y 0,50 kg, y una correa nueva de eBay de USD 30 y 0,30 kg. Una unidad de cada producto; precios finales con TAX de USD 0 incluido.","Corrijo solo el precio de la correa: es USD 38 finales. Los demás datos no cambian.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT024","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH05","mensajes":["Soy Ximena Paredes. Quiero cotizar un libro impreso nuevo de Amazon de USD 28 y 0,60 kg, y una billetera nueva de eBay de USD 40 y 0,40 kg. Una unidad de cada producto; precios finales con TAX de USD 0 incluido.","Elimina la billetera de la cotización. Me quedo solo con el libro y sus mismos datos.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT025","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH05","mensajes":["Soy Yolanda Prieto. Quiero cotizar un libro impreso nuevo de Amazon de USD 25 y una billetera nueva de eBay de USD 35, una unidad de cada producto. Ambos precios son finales con TAX confirmado de USD 0 incluido.","El pedido completo pesa 1,30 kg con embalaje. No tengo los pesos individuales.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT026","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH06","mensajes":["Soy Zoe Lara. Quiero cotizar una billetera nueva de Amazon, una unidad por USD 42 finales con TAX confirmado de USD 0 incluido.","Antes de seguir, ¿cuál es la dirección del almacén de Miami?","La billetera pesa 0,60 kg con embalaje.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT027","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH06","mensajes":["Soy Alicia Vega. Quiero cotizar un libro impreso nuevo de eBay, una unidad por USD 35 finales con TAX confirmado de USD 0 incluido y peso 0,70 kg con embalaje. Prefiero recogerlo personalmente en Lima.","¿Qué significa GX en la dirección?","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT028","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH06","mensajes":["Soy César Melo. Quiero cotizar un libro impreso nuevo de Amazon, una unidad de USD 45 finales con TAX confirmado de USD 0 incluido. Pesa 750 gramos con embalaje.","Quiero que un asesor revise estos datos y continúe con mi cotización."]},{"id":"CHAT029","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH06","mensajes":["Soy Diana Solís. Quiero cotizar una billetera nueva de eBay, una unidad por USD 58 finales con TAX confirmado de USD 0 incluido y peso 0,60 kg con embalaje. Todavía no he comprado ni pagado nada.","Quiero que un asesor revise esos datos antes de decidir si compro."]},{"id":"CHAT030","banco":"CHATS_COMPARTIDOS","banco_id":"CHATS_RECOP_RESUMEN_30_V1","version_banco":"GLOBEX_EVALUACION_CONSOLIDADA_20261115_V1","universo":"CH06","mensajes":["Soy Esteban Cruz. Quiero cotizar una mochila textil nueva de Amazon, una unidad por USD 70 finales con TAX confirmado de USD 0 incluido y peso 1,10 kg con embalaje.","Antes de continuar, dime qué producto, tienda, precio, cantidad y peso tienes registrados.","Quiero que un asesor revise estos datos y continúe con mi cotización."]}]}'
assert hashlib.sha256(CODIGO_EJECUTOR.encode("utf-8")).hexdigest() == 'db645f2aad7d703bc5829daf83b12fe3875a9748e4f1bf518f47fa7855231367'
assert hashlib.sha256(CODIGO_EXPORTADOR.encode("utf-8")).hexdigest() == '989c8eb2e034a65abac89c709b2ed810efa5c1a6fd96b85065c39298346adb2f'
exportador = types.ModuleType("globex_export")
exportador.__file__ = str(Path.cwd() / "globex_export.py")
sys.modules[exportador.__name__] = exportador
exec(compile(CODIGO_EXPORTADOR, exportador.__file__, "exec"), exportador.__dict__)
globex = types.ModuleType("globex_ejecutor_final")
globex.__file__ = str(Path.cwd() / "globex_ejecutor_final.py")
sys.modules[globex.__name__] = globex
exec(compile(CODIGO_EJECUTOR, globex.__file__, "exec"), globex.__dict__)
BANCO = json.loads(BANCO_JSON)
globex.validate_bundle(BANCO)
assert len(BANCO["casos"]) == 228
print("Entorno listo. Python", sys.version.split()[0], "| Casos:", len(BANCO["casos"]))
NUEVA_CORRIDA = False
resultado = globex.run(
    BANCO, perfil="todos", nueva_corrida=NUEVA_CORRIDA,
    output_base=globex.notebook_output_dir(False),
)
globex.notebook_download(resultado)
if sys.platform == "win32" and resultado.get("zip"):
    import os
    try:
        os.startfile(resultado["zip"])
    except OSError:
        print("Archivo terminado; usa el enlace de resultados mostrado arriba.")
